# C5. ROC và precision–recall khi tỷ lệ lớp dương giảm từ 50% xuống 1%

Điểm theo mô hình hai phân phối chuẩn: điểm | y=0 ~ N(0, 1), điểm | y=1 ~ N(μ, 1), μ = 1,5 (không đổi theo π).
n = 20 000 mỗi tập, π ∈ {50%, 20%, 10%, 5%, 1%}, seed 42–46. ROC/PR/AUC/AP tính bằng sklearn (được phép theo RULES).
Đường lý thuyết: TPR(t) = 1 − Φ(t − μ), FPR(t) = 1 − Φ(t), precision(t) = π·TPR / (π·TPR + (1 − π)·FPR).

In [1]:
import os, sys, warnings
import numpy as np
import pandas as pd
ROOT = os.path.abspath("..")
sys.path.insert(0, os.path.join(ROOT, "src"))
RES = os.path.join(ROOT, "results")
os.makedirs(RES, exist_ok=True)
SEEDS = [42, 43, 44, 45, 46]

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve

MU, N = 1.5, 20_000
PIS = [0.5, 0.2, 0.1, 0.05, 0.01]
t_grid = np.linspace(-8, 10, 20_001)
TPR_th, FPR_th = norm.sf(t_grid - MU), norm.sf(t_grid)
AUC_th = norm.cdf(MU / np.sqrt(2))
T80 = MU + norm.ppf(0.2)                      # ngưỡng có TPR lý thuyết = 0,8


def prec_th(pi):
    return pi * TPR_th / (pi * TPR_th + (1 - pi) * FPR_th)


def ap_th(pi):
    """AP lý thuyết = ∫ precision d(recall) (recall = TPR giảm theo t)."""
    p = prec_th(pi)
    return float(np.sum(-np.diff(TPR_th) * (p[1:] + p[:-1]) / 2))


def sample(pi, seed):
    rng = np.random.default_rng(seed)
    y = (rng.uniform(size=N) < pi).astype(int)
    s = rng.normal(size=N) + MU * y
    return s, y

In [2]:
rows = []
for pi in PIS:
    for seed in SEEDS:
        s, y = sample(pi, seed)
        d = s >= T80
        tp, fp = np.sum(d & (y == 1)), np.sum(d & (y == 0))
        rows.append(dict(pi=pi, seed=seed, n_duong=int(y.sum()),
                         auc_roc=roc_auc_score(y, s), ap=average_precision_score(y, s),
                         tpr_t80=tp / y.sum(), fpr_t80=fp / (y == 0).sum(), precision_t80=tp / max(tp + fp, 1)))
raw = pd.DataFrame(rows)
agg = raw.groupby("pi").agg(["mean", "std"]).drop(columns="seed").sort_index(ascending=False)
th = pd.DataFrame({"pi": PIS, "auc_roc_ly_thuyet": AUC_th, "ap_ly_thuyet": [ap_th(p) for p in PIS],
                   "precision_t80_ly_thuyet": [0.8 * p / (0.8 * p + (1 - p) * norm.sf(T80)) for p in PIS]})
flat = agg.copy(); flat.columns = [f"{a}_{b}" for a, b in flat.columns]
out = flat.reset_index().merge(th, on="pi")
out.to_csv(os.path.join(RES, "C5_roc_pr.csv"), index=False)
print(f"AUC lý thuyết = {AUC_th:.4f}; ngưỡng t80 = {T80:.4f}; FPR lý thuyết tại t80 = {norm.sf(T80):.4f}")
out.round(4)

AUC lý thuyết = 0.8556; ngưỡng t80 = 0.6584; FPR lý thuyết tại t80 = 0.2551


,pi,n_duong_mean,n_duong_std,auc_roc_mean,auc_roc_std,ap_mean,ap_std,tpr_t80_mean,tpr_t80_std,fpr_t80_mean,fpr_t80_std,precision_t80_mean,precision_t80_std,auc_roc_ly_thuyet,ap_ly_thuyet,precision_t80_ly_thuyet
0,0.50,9990.6,77.5841,0.8548,0.0029,0.8512,0.0047,0.8002,0.0032,0.2561,0.0063,0.7572,0.0065,0.8556,0.8538,0.7582
1,0.20,4015.8,10.4259,0.8554,0.0027,0.6375,0.0051,0.8000,0.0040,0.2542,0.0034,0.4416,0.0034,0.8556,0.6387,0.4394
2,0.10,1996.8,28.6479,0.8559,0.0052,0.4723,0.0088,0.8020,0.0098,0.2546,0.0028,0.2590,0.0054,0.8556,0.4781,0.2584
3,0.05,993.6,21.8358,0.8520,0.0086,0.3310,0.0105,0.7881,0.0167,0.2540,0.0028,0.1396,0.0043,0.8556,0.3354,0.1416
4,0.01,204.4,7.8930,0.8509,0.0113,0.1097,0.0147,0.7710,0.0281,0.2542,0.0029,0.0304,0.0013,0.8556,0.1155,0.0307


In [3]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
colors = plt.cm.viridis(np.linspace(0, 0.9, len(PIS)))
for c, pi in zip(colors, PIS):
    s, y = sample(pi, 42)
    fpr, tpr, _ = roc_curve(y, s)
    prec, rec, _ = precision_recall_curve(y, s)
    ax1.plot(fpr, tpr, color=c, lw=1.5, label=f"π = {pi:.0%} (AUC = {roc_auc_score(y, s):.3f})")
    ax2.plot(rec, prec, color=c, lw=1.5, label=f"π = {pi:.0%} (AP = {average_precision_score(y, s):.3f})")
    ax2.plot(TPR_th, prec_th(pi), color=c, lw=1, ls="--")
    ax2.axhline(pi, color=c, lw=0.8, ls=":")
ax1.plot(FPR_th, TPR_th, "k--", lw=1, label=f"lý thuyết (AUC = {AUC_th:.3f})")
ax1.plot([0, 1], [0, 1], color="gray", lw=0.8, ls=":")
ax1.set(xlabel="FPR", ylabel="TPR (recall)", title="(a) ROC", xlim=(0, 1), ylim=(0, 1.01))
ax2.set(xlabel="Recall", ylabel="Precision", title="(b) Precision–recall (đứt: lý thuyết; chấm: π)",
        xlim=(0, 1), ylim=(0, 1.01))
ax1.legend(fontsize="small", loc="lower right"); ax2.legend(fontsize="small", loc="upper right")
fig.tight_layout()
fig.savefig(os.path.join(RES, "C5_roc_pr.png"), dpi=150)
plt.close(fig)
print("Đã lưu results/C5_roc_pr.png")

Đã lưu results/C5_roc_pr.png
